# سبق 18: کرپٹوگرافک رسیدوں کے ساتھ AI ایجنٹس کی حفاظت

## عملی نوٹ بک

یہ نوٹ بک چار مشقوں کے ذریعے چلتی ہے:

1. **اپنی پہلی رسید پر دستخط کریں** ایک ایجنٹ ٹول کال کے لیے اور اس کی تصدیق کریں۔
2. **رسید میں چھیڑ چھاڑ کریں** اور تصدیق ناکام ہوتے دیکھیں۔
3. **تین رسیدوں کا زنجیر بنائیں** اور زنجیر کی سالمیت کی تصدیق کریں۔
4. **مائیکروسافٹ ایجنٹ فریم ورک ٹول کال کو لپیٹیں** تاکہ ہر عمل ایک رسید جاری کرے۔

تمام کرپٹوگرافک پرمیٹوز کا درآمد اچھے طریقے سے سنبھالی گئی لائبریریوں سے ہوتا ہے (`pynacl` برائے Ed25519، `jcs` برائے RFC 8785 کینونیکل JSON، `hashlib` پائتھن کے معیاری لائبریری سے SHA-256 کے لیے)۔ رسید کا منطق خود سادہ پائتھن ہے جسے آپ پڑھ اور ترمیم کر سکتے ہیں۔

خلیات کو ترتیب وار چلائیں۔ ہر سیکشن مختصر اور خود مختار ہے۔


## سیٹ اپ

دو انحصار انسٹال کریں۔ دونوں کے لائسنس اجازت یافتہ ہیں (Apache-2.0 / MIT)۔


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## معاون یوٹیلٹیز

یہ دو معاون base64url انکوڈنگ (بغیر پیڈنگ کے) اور کسی بھی اشیاء کے SHA-256 ہیشنگ کو سنبھالتے ہیں۔ یہ نوٹ بک کے باقی حصے کو صرف رسید کے منطق پر مرکوز رکھتے ہیں۔


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## سیکشن 1: اپنا پہلا رسید سائن کریں

تصور کریں کہ ہمارے ایجنٹ برائے **Contoso Travel** نے ابھی کسٹمر کے لئے سڈنی سے لاس اینجلس کی فلائٹس دیکھیں۔ ہم اس ٹول کال کو ایک دستخط شدہ رسید کے طور پر ریکارڈ کرنا چاہتے ہیں تاکہ مستقبل کا آڈیٹر اس کی تصدیق کر سکے بغیر ہم پر اعتماد کیے۔

### قدم 1.1: دستخط کے لیے کلید تیار کریں

پروڈکشن میں، ایجنٹ کی دستخط کی کلید ہارڈویئر سیکیورٹی ماڈیول (HSM)، Azure Key Vault، یا کسی مشابہ محفوظ اسٹور میں محفوظ ہوتی ہے۔ اس سبق کے لیے ہم میموری میں نئی کلید تیار کرتے ہیں۔


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: رسید کا پیلوڈ تیار کریں

پیلوڈ میں وہ سب کچھ شامل ہوتا ہے جس کی رسید تصدیق کرنا چاہتی ہے: کس نے عمل کیا، کون سا ٹول استعمال کیا، کون سے دلائل دیے، کیا نتیجہ واپس آیا، کس پالیسی کے تحت، اور کب۔ ہم دلائل اور نتائج کا ہیش بناتے ہیں بجائے اس کے کہ انہیں براہ راست شامل کریں تاکہ رسید حساس مواد نہ لیک کرے۔


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: رسید پر دستخط کریں اور اسے اکٹھا کریں

تین مراحل:

1. JCS استعمال کرتے ہوئے پے لوڈ کو کینانکلائز کریں تاکہ دو نفاذ جو ایک ہی منطقی رسید تیار کرتے ہیں، بائٹ کے اعتبار سے بالکل یکساں بائٹس پیدا کریں۔
2. کینانکل بائٹس کو SHA-256 کے ساتھ ہیش کریں۔
3. ہیش پر Ed25519 پرائیوٹ کلید سے دستخط کریں۔

دستخط کو پھر اصل پے لوڈ کے ساتھ منسلک کیا جاتا ہے تاکہ حتمی رسید تیار کی جا سکے۔


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### قدم 1.4: رسید کی توثیق کریں

توثیق عمل کو الٹ دیتی ہے۔ ہم دستخط کو ہٹا دیتے ہیں، معیاری ہیش کو دوبارہ حساب کرتے ہیں، اور دستخط کو رسید میں موجود عوامی کلید کے مطابق چیک کرتے ہیں۔

ایسا آڈیٹر جو یہ توثیق کر رہا ہو، اسے ہم سے کچھ بھی نہیں چاہیے سوائے خود رسید کے۔ نہ کسی سروس کو کال کرنا ہے، نہ کسی کلید کی ڈائریکٹری سے دریافت کرنا ہے، اور نہ ہی کوئی اعتماد درکار ہے۔


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

تو آپ کو `Receipt is valid: True` نظر آنا چاہیے۔ ایجنٹ نے اپنی پہلی کرپٹوگرافکلی دستخط شدہ آڈٹ ریکارڈ تیار کر لی ہے۔


## سیکشن 2: رسید میں چھیڑ چھاڑ کریں

رسیدوں کا پورا مقصد یہ ہے کہ وہ چھیڑ چھاڑ کے ثبوت دیتے ہیں۔ آئیے اس کا ثبوت پیش کرتے ہیں۔

ہم رسید کے بالکل ایک کردار کی ترمیم کریں گے اور تصدیق کو ناکام ہوتے دیکھیں گے۔


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### کیا ہوا؟

جب ہم نے `policy_id` کو تبدیل کیا، تو canonical bytes تبدیل ہو گئے۔ ان bytes کا SHA-256 ہیش بھی تبدیل ہو گیا۔ دستخط (جو اصل ہیش پر تھا) اب نئے ہیش سے میل نہیں کھاتا۔ توثیق صحیح طریقے سے `False` واپس کرتی ہے۔

رسید کا کوئی بھی فیلڈ تبدیل کر کے اسے توثیق کرنا ممکن نہیں جب تک کہ حملہ آور کے پاس نجی کلید نہ ہو۔ جب تک نجی کلید کسی کلید بھنڈر (key vault) میں محفوظ ہے اور پبلک کلید شائع ہو چکی ہے، چھیڑ چھاڑ کو چھپانا ناممکن ہے۔

خود کوشش کریں: اوپر والے سیل میں `tool_name` یا `agent_id` یا `timestamp` کو تبدیل کریں اور دوبارہ چلائیں۔ ہر تبدیلی ایک غیر معتبر رسید پیدا کرتی ہے۔


## حصہ 3: رسیدوں کو ایک ساتھ جوڑنا

ایک واحد رسید ایک عمل کی حفاظت کرتی ہے۔ زیادہ تر ایجنٹ کئی اقدامات کرتے ہیں۔ پورے سلسلے کو مداخلت سے محفوظ بنانے کے لیے، ہم ہر رسید کو پچھلی رسید سے جوڑتے ہیں، اس طرح کہ نئی رسید کے پے لوڈ میں پچھلی رسید کا ہیش شامل کیا جاتا ہے۔

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

اگر کوئی رسید ہٹا دے یا اس کی ترتیب بدل دے، تو چین بالکل اسی نقطہ پر ٹوٹ جاتی ہے۔ کسی بھی بعد کی رسید کی تصدیق ناکام ہو جاتی ہے کیونکہ اس کا `previous_receipt_hash` اب اپنے پیش رو کے اصل ہیش سے مماثل نہیں رہتا۔


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

اب درمیان کی رسید میں چھیڑ چھاڑ کر زنجیر کو توڑیں اور دوبارہ تصدیق کریں۔ چھیڑی گئی رسید اپنی دستخط کی جانچ میں ناکام ہو جاتی ہے، اور اگلی رسید اپنی زنجیر-لنک کی جانچ میں ناکام ہو جاتی ہے (کیونکہ اس کا `previous_receipt_hash` اب تبدیل شدہ درمیانی رسید کے ہیش سے میل نہیں کھاتا)۔


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

رسید 0 ابھی بھی تصدیق کرتا ہے (اس میں کوئی ترمیم نہیں کی گئی اور اس کا کوئی پیش رو نہیں ہے جس پر انحصار کیا جا سکے)۔ رسید 1 اپنی دستخط کی جانچ میں ناکام ہو جاتی ہے کیونکہ ہم نے `tool_args_hash` تبدیل کیا ہے۔ رسید 2 اپنی چین-لنک چیک میں ناکام ہو جاتی ہے کیونکہ اس کا `previous_receipt_hash` اصلی (اب ترمیم شدہ) رسید 1 کے خلاف حساب کیا گیا تھا۔

یہاں تک کہ اگر حملہ آور ترمیم شدہ رسید 1 پر دوبارہ دستخط بھی کر دے (جو وہ پرائیویٹ کلید کے بغیر نہیں کر سکتے)، رسید 2 میں چین-لنک کی عدم مطابقت پھر بھی چالاکی کو بے نقاب کرے گی۔ تبدیلی کو چھپانے کے لیے، حملہ آور کو ترمیم کے نقطہ آغاز سے ہر رسید پر دوبارہ دستخط کرنا پڑے گا، جس کے لیے پرائیویٹ کلید کا قبضہ ضروری ہے۔


## سیکشن 4: ایک ایجنٹ ٹول کال کو رسید کی دستخط کے ساتھ لپیٹیں

حقیقی تعیناتی میں، آپ نہیں چاہتے کہ ہر ایجنٹ مصنف کو `make_receipt` کال کرنا یاد رہے۔ آپ چاہتے ہیں کہ ہر ٹول کی کال پر رسید کی دستخط خودکار ہو۔

یہاں سب سے سادہ طریقہ ہے: ایک ریپر کلاس جو کوئی بھی کال کرنے والا ٹول فنکشن لے اور اس کی رسید جاری کرنے والی ورژن واپس دے۔ یہ کسی بھی ایجنٹ فریم ورک کے ساتھ مطابقت رکھتا ہے، بشمول Microsoft Agent Framework (`agent_framework.azure`)۔

اگر آپ کے پاس Azure AI Foundry پروجیکٹ سیٹ اپ نہیں ہے، تو نیچے دیا گیا مقامی موک بھی اس طریقہ کار کی وضاحت کرتا ہے۔


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ انضمام

اوپر دیا گیا `ReceiptedTool` ریپر فریم ورک سے آزاد ہے۔ اسے مائیکروسافٹ ایجنٹ فریم ورک کے اندر استعمال کرنے کے لیے، لپیٹے گئے فنکشن کو ٹول کے طور پر رجسٹر کریں۔ ایک خاکہ (آپ موک کو حقیقی Azure AI Foundry ٹول رجسٹریشن سے تبدیل کریں گے):

```python
# پسیوڈو کوڈ جو انٹیگریشن کی شکل دکھاتا ہے۔
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="آپ ایک کونٹوسو ٹریول ایجنٹ ہیں ...",
#     tools=[receipted_lookup],   # لپٹے ہوئے ٹول، کچا فنکشن نہیں
# )
# response = agent.run("جون میں سڈنی سے لاس اینجلس کے لیے پروازیں تلاش کریں۔")
#
# # رن کے بعد، ہر ٹول کال جسے ایجنٹ نے کیا ہے ایک دستخط شدہ رسید ہوتی ہے:
# audit_chain = receipted_lookup.receipts
```

ایجنٹ فریم ورک کو رسیدوں کے بارے میں کچھ جاننے کی ضرورت نہیں ہے۔ رسید کے دستخط ٹول کے گرد لپیٹے گئے ہیں، فریم ورک میں شامل نہیں ہیں۔ اسی طرح آپ موجودہ ایجنٹ کوڈ میں دوبارہ لکھے بغیر ماخذ شامل کرتے ہیں۔


## خلاصہ اور اضافی چیلنج

آپ نے:

- ایک Ed25519 کلیدی جوڑا تیار کیا ہے۔
- ایک ایجنٹ ٹول کال کے لیے رسید بنائی اور دستخط کیے۔
- رسید کو آف لائن صرف پبلک کی کی مدد سے تصدیق کی۔
- ایک رسید میں چھیڑ چھاڑ کی اور تصدیق ناکام ہوتی دیکھی۔
- تین رسیدوں کی ہیش چین والی ترتیب بنائی۔
- چین کے بیچ میں چھیڑ چھاڑ کی اور دستخط اور چین-لنک دونوں کی ناکامی دیکھی۔
- ایک ٹول فنکشن کو خودکار رسید دستخط کے ساتھ لپیٹ دیا۔

**اضافی چیلنج۔** رسید کے اسکیمے میں `request_id` فیلڈ (جو تقسیم شدہ ٹریسنگ کے لیے UUID ہے) شامل کریں۔ `make_receipt` کو اپ ڈیٹ کریں تاکہ وہ اسے شامل کرے، اور تصدیق کریں کہ رسیدیں ابتدا سے آخر تک تصدیق ہو رہی ہیں۔ پھر دستخط کے بعد اس فیلڈ میں ترمیم کریں اور تصدیق ناکام ہو جانا یقینی بنائیں۔ اس سے آپ کو یہ سمجھنا پڑے گا کہ canonical encoding کے ہر بائٹ کا دستخط میں کیا کردار ہے۔

**اہم حد بندی۔** رسیدیں تین چیزیں اور صرف تین چیزیں ثابت کرتی ہیں: صفات (اس کلید نے یہ مواد دستخط کیا ہے)، سالمیت (مواد دستخط کے بعد تبدیل نہیں ہوا)، اور ترتیب (یہ رسید اس رسید کے بعد آئی ہے)۔ یہ ثابت نہیں کرتیں کہ ایجنٹ کا عمل درست تھا، کہ `policy_id` میں نامزد پالیسی کا واقعی جائزہ لیا گیا، یا کہ ایجنٹ نے ہر قاعدہ کی پیروی کی۔ رسیدیں ایک بنیاد ہیں۔ گورننس آپ کا بنایا ہوا نظام ہے۔

سبق کا README اس حد بندی کو ذہن میں رکھتے ہوئے دوبارہ پڑھیں۔ رسیدوں کے ساتھ ٹیموں کی عام غلطی یہ سمجھنا ہے کہ "ہمارے پاس رسیدیں ہیں" مطلب "ہم گورنڈ ہیں"۔ ایسا نہیں ہوتا۔ رسیدیں ایجنٹ کے رویے کی آڈٹ سازی کرتی ہیں۔ یہ اسے درست نہیں بناتیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
